# Data Validation

Often when we write classes with attributes, we expect these attributes to have certain values. For example in our `Pokemon` class, we saw that its `hp` attribute should never be smaller than `0`. If its hitpoints would drop below `0` by taking damage, it should be set to `0` instead.

In the previous exercise we handled this by including this piece of code in the `attack()` method:

```python .noeval
other.hp -= move.damage
if other.hp < 0:
    other.hp = 0
```

And this works fine. But imagine there are multiple places in the code where we have to change the `hp` of the pokemon, which will indeed be the case if we would try to build a fully-fledged pokemon game. In that case we would have to add this condition in all of these places. And if we would forget to do it in one of them, pokemon might get negative `hp` which could introduce bugs in other parts of our code.

This kind of situation is incredibly common. Other examples of this could be:

- an `email_address` attribute that should always be in lowercase (so we can easily compare it to other email addresses), so whenever setting a new value we should `.lower()` that value (eg `'Alice@gmail.com'` would be converted to `'alice@gmail.com'`)
- any `String` attribute whose values are provided through `input()` and could therefore start or end with accidental whitespace like `' '` or `'\n'` that should be stripped with `.strip()` before assigning that value (eg `"Alice \n"` should become `"Alice"`)
- a `price` attribute that takes a values that should always be rounded to 2 decimal places (eg `3.452` becomes `3.45`)

Whenever we set the values of these attributes, we want these operations to be done consistently

## 'setter' methods

The most straightforward way to make sure that these operations are done every time you assign a value, is to **create a method** that **sets the value** in this way and **use this method every time** you want to change the value of its attribute.

For example for the `Pokemon` class we could create the method `set_hp()`:

In [ ]:
class Pokemon:
    def __init__(self, name, hp, moves):
        self.name = name
        self.moves = moves
        self.hp = hp
        self.max_hp = hp

    def set_hp(self, new_hp):
        self.hp = new_hp
        if self.hp < 0:
            self.hp = 0

Now, every time we want to change the value of the attribute `hp`, instead of just assigning a new value to it, like we did before:

In [ ]:
pokemon.hp = pokemon.hp - damage

we use our new method:

In [ ]:
pokemon.set_hp(pokemon.hp - damage)

**If we do this consistently**, we are sure **`hp` will never be lower than `0`**

Or for the case of wanting `email` addresses to be lower case, we could have the class:

In [ ]:
class Person:
    def __init__(self, name, email):
        self.name = name
        self.email = email

    def set_email(self, email):
        self.email = email.lower()

james = Person("James Bond", 'jamesbond@gmail.com')
james.set_email("JamesBond@ucll.be")
print(james.email)

Creating **this kind of method**, whose only purpose it is **to set the value** of some attribute, is so incredibly common that they have a specific name: **setter methods**.

## Raising Errors

In the previous sections we talked about using **setter methods** to automatically convert 'incorrect' values to a valid value. This is very useful in cases where we **can reasonably assume what the value should be** converted to:

- In the case of `Pokemon` `hp`, we can assume that if we try to set a negative value, it should be capped at 0.
- When a user inputs the string `Alice  '` when we ask their name, we can reasonably assume they meant to enter `'Alice'` (without spaces)

However, this is **not always the case**.

For example, a programmer creates a list with a couple of elements. They then try to get an element from that list using `my_list[20]` but it turns out the list `my_list` only has `10` elements. Should Python just assume that if the programmer enters `20` while there are only `10` elements, they probably meant to select the closest element to what they entered (in this case the 10th element)? As you probably know by now, Python does not do this:

In [ ]:
my_list = list(range(10))
print(my_list[20])

Instead, **Python lets the code crash**. **Why?**

Well, if you try to select the 20th element from a list, Python assumes that you think this list has a least 20 items. And the fact that it doesn't, means that something probably already went wrong earlier in the code (because if nothing went wrong, the list \**would have* had (at least) 20 items).

Of course, Python could have been programmed to let this go on for longer, to simply let the code run for as long is it could (If you have experience with JavaScript you know how fun that is). However doing that typically results in programs that simply produce wrong output (because there was in fact a logical mistake somewhere in the code). And then you as a programmer would have to go search through your code and try to find the point in your program where it all went wrong. (which as you've probably already experienced by now is not easy)

So instead, Python doesn't let it go on. It stops the program and says "This is wrong, start looking for your mistake here"

We can do the same thing.

### Validating `PlayingCard`s

Let's take a look at a different example, the `PlayingCard` class we wrote a couple of classes ago.

In [ ]:
class PlayingCard:
    def __init__(self, string_value):
        self.suit = string_value[0]
        self.rank = string_value[1:]
        self.value = string_value

ace_of_spades = PlayingCard("SA")
print(ace_of_spades.suit)

As you might remember, the string that we pass to the `PlayingCard` constructor, should be in a specific format:

- the first character should be either `'H'` for hearts, `'S'` for spades, `'C'` for clubs or `'D'` for diamonds
- the rest of the string should be `'1'`-`'10'`

So what if while using this class, we accidentally type something different? For example, we accidentally switch the characters around:

In [ ]:
ace_of_spaces = PlayingCard("AS")
print(ace_of_spaces.suit) # the suit is Ace
print(ace_of_spaces.rank) # the rank is Spades

Clearly this is wrong, and having this card in the game we are programming will almost certainly result in some bug or crash later on in the code. But for now it doesn't, creating the card doesn't let the program crash.

But wouldn't we prefer that the program did already crash here? If the program would crash right here, at least it would be very obvious what our mistake was. Then we can fix it, and run the program again.

Situations like this are also very common. Other examples would be:

- a `day_of_week` attribute that should only take the values `'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'`
- a `birthyear` attribute that should be a positive integer
- ...

So let's see how we can do this.

### `raise ValueError()`

So how can we let our program crash? Well the syntax is very simple. Simply run the following cell: (it should result in an error)

In [ ]:
raise ValueError()

As you can see we get exactly the error that we wrote: `ValueError`.

This statement consist of two parts:

- the keyword `raise` which tells python that we want to "raise an error", meaning that we want to crash to program
- `ValueError()` creates the exact error object we want to raise.

There are a lot of different kinds of errors that we could raise. In the previous section we saw that Python raises an `IndexError` when we try to access an element in a list at an index that doesn't exist. We can raise this kind of Error as well:

In [ ]:
raise IndexError()

At this point you might notice one small difference between the `IndexError` that we just raised and the one that Python raised. When Python does it, we see:

```none
IndexError: list index out of range
```

while ours just says:

```none
IndexError:
```

(without the text `list index out of range`)

That's because when raising an error, you can provide a little error message to make it more clear why the error was raised:

In [ ]:
raise IndexError("list index out of range")

or with `ValueError`

In [ ]:
raise ValueError("Oh no! we did something wrong.. Don't worry, you'll figure it out, I believe in you <3")

While it's not required, passing a specific message when raising an error helps out a lot when trying to find your mistakes quickly.

So what's the difference between `ValueError` and `IndexError`? In actuality nothing that matters right now, we typically use different 'kinds' of errors to differentiate different things that could go wrong in the code:

- `IndexError` means that something went wrong when indexing something
- `ValueError` generally means that a wrong value was passed

but this is not something you should really be concerned with right now. When we raise errors, we are **only going to use `ValueError`**

### `ValueError` in `PlayingCard`

So, now that we know how to raise an error, let's put it into practice to make sure we can't pass 'wrong' arguments to the `PlayingCard` constructor:

As a reminder, we previously had the class:

In [ ]:
class PlayingCard:
    def __init__(self, string_value):
        self.suit = string_value[0]
        self.rank = string_value[1:]
        self.value = string_value

ace_of_spades = PlayingCard("AS")
print(ace_of_spades.suit)

where it was possible to create this object, even though it's an invalid playing card. Instead we could do:

In [ ]:
class PlayingCard:
    def __init__(self, string_value):
        suit = string_value[0]
        if suit not in ["H", "D", "C", "S"]:
            raise ValueError(f"'{suit}' is not a valid suit: use 'H', 'D', 'C' or 'S' instead")
        rank = string_value[1:]
        if rank not in ["A", "2", "3", "4", "5", "6", "7", "8", "9", "10", "J", "Q", "K"]:
            raise ValueError(f'"{rank}" is not a valid rank: use "A", "2"-"10", "J", "Q" or "K" instead')
        self.suit = suit
        self.rank = rank
        self.value = string_value

ace_of_spades = PlayingCard("AS")
print(ace_of_spades.suit)

If we run this code, we can observe that the first condition is immediately violated, so the first error is raised (the code afterwards is not executed - because the program stops - which is why the second error can't also be raised)

of course, if we try to create an object where only the rank is wrong, the second error will be raised:

In [ ]:
twelve_of_clubs = PlayingCard("C12") # 12 is not a valid rank

## Raising errors in setters

Earlier in this chapter we used 'setter' methods to consistently convert values to a specific format before assigning them to an attribute. However, when the value being passed to the setter can't be converted to something useful, we can opt to raise an error instead. For example, consider this `Course` class that represents a course you can take at school:

In [ ]:
class Course:
    def __init__(self, name, day_of_week, slot):
        self.name = name
        self.day_of_week = day_of_week
        self.slot = slot

programming_1 = Course("Programming 1", "Monday", 3)

where:

- `name` represents the name of the course (`"Programming 1"`)
- `day_of_week` of course represents the day of the week the course is taught (`"Monday"`, `"Tuesday"`, `"Wednesday"`, `"Thursday"` or `"Friday"`)
- `slot` represents the time the course is taught. At UCLL we have 4 slots 8h-10h, 10h-12h, 13h-15h and 15h-17h that we simply numbers `1`, `2`, `3` and `4`

but as we know, right now nothing is preventing us from changing these values to something invalid:

In [ ]:
programming_1.day_of_week = "Every day"

To prevent this, we can add setter methods that will raise errors when we do something like this:

In [ ]:
class Course:
    def __init__(self, name, day_of_week, slot):
        self.name = name
        self.day_of_week = day_of_week
        self.slot = slot

    def set_day_of_week(self, new_day_of_week):
        if new_day_of_week not in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]:
            raise ValueError(f'"{new_day_of_week}" is not a weekday, choose "Monday", "Tuesday", "Wednesday", "Thursday" or "Friday"')
        self.day_of_week = new_day_of_week

    def set_slot(self, new_slot):
        if new_slot not in [1, 2, 3, 4]:
            raise ValueError(f'{new_slot} is not a valid slot, choose 1 (8h-10h), 2 (10h-12h), 3 (13h-15h) or 4 (15h-17h) instead')
        self.slot = new_slot

programming_1 = Course("Programming 1", "Monday", 3)

If we then try to change one of the attributes to an invalid value, we will get an error instead:

In [ ]:
programming_1.set_day_of_week("Every day")

while if use a correct value, it will still work:

In [ ]:
programming_1.set_day_of_week("Thursday")
print(programming_1.day_of_week)

## Using our setters in the constructor

Using the approach of doing the validation in the setter, of course still allows us to assign incorrect values in the constructor, for example:

In [ ]:
programming_2 = Course("Programming 2", "Every day", 10)
print(programming_2.day_of_week)
print(programming_2.slot)

Therefore, if there is a setter, we almost always want to use it in the constructor as well, like so

In [ ]:
class Course:
    def __init__(self, name, day_of_week, slot):
        self.name = name
        self.set_day_of_week(day_of_week)
        self.set_slot(slot)

    def set_day_of_week(self, new_day_of_week):
        if new_day_of_week not in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]:
            raise ValueError(f'"{new_day_of_week}" is not a weekday, choose "Monday", "Tuesday", "Wednesday", "Thursday" or "Friday"')
        self.day_of_week = new_day_of_week

    def set_slot(self, new_slot):
        if new_slot not in [1, 2, 3, 4]:
            raise ValueError(f'{new_slot} is not a valid slot, choose 1 (8h-10h), 2 (10h-12h), 3 (13h-15h) or 4 (15h-17h) instead')
        self.slot = new_slot

If we now pass faulty values during initialization, our code raises an error immediately:

In [ ]:
programming_2 = Course("Programming 2", "Every day", 10)

> Note: at first it may be confusing that there is no longer a line such as `self.day_of_week = ...` in the constructor. How is the attribute `day_of_week` created in this case? Well, when we call `self.set_day_of_week(day_of_week)` in the constructor the code immediately jumps to that method where, after the validation, we see that `self.day_of_week = ...` still creates the attribute.

## Exercises

### Exercise 81.1: Clean contacts

You're writing an address book application. Users type in email addresses by hand, and as you know by now, users type all kinds of things: `" Alice@Gmail.Com "`, `"BOB@ucll.be\n"`, ... To be able to compare and look up email addresses reliably, we want every stored address to be **stripped of surrounding whitespace** and **entirely in lowercase**.

Write a class `Contact` with the attributes `name` and `email`. Add a setter method `set_email(new_email)` that cleans up the value before storing it, and make sure this cleanup also happens when the object is created.

```python .noeval
contact = Contact("Alice", " Alice@Gmail.Com ")
print(contact.email) # alice@gmail.com

contact.set_email("Alice@UCLL.be\n")
print(contact.email) # alice@ucll.be
```

Implement this class in the file [exercise_81_1_email_setter.py](concept-exercises/exercise_81_1_email_setter.py).
Run the following cell to verify that your implementation is correct:

In [ ]:
# code to run the tests
!python3 -m pytest -q --tb=short concept-exercises/.tests/test_exercise_81_1_email_setter.py

### Exercise 81.2: Impossible grades

At UCLL, exam grades always go from `0` to `20`. A grade of `25` or `-3` simply doesn't exist, and if one ends up in the system, something has clearly gone wrong somewhere — better to find out immediately than after the results have been sent out.

Write a class `ExamResult` with the attributes `course` and `grade`. Add a setter method `set_grade(new_grade)` that raises a `ValueError` when the new grade is not between `0` and `20` (both included), and make sure this validation also happens when the object is created.

```python .noeval
result = ExamResult("Programming 1", 14)
print(result.course) # Programming 1
print(result.grade)  # 14

result.set_grade(16)
print(result.grade)  # 16

result.set_grade(25)  # ValueError
```
and creating an invalid result should also be impossible:
```python .noeval
result = ExamResult("Programming 1", -3)  # ValueError
```

Implement this class in the file [exercise_81_2_grade_validation.py](concept-exercises/exercise_81_2_grade_validation.py).
Run the following cell to verify that your implementation is correct:

In [ ]:
# code to run the tests
!python3 -m pytest -q --tb=short concept-exercises/.tests/test_exercise_81_2_grade_validation.py